<a href="https://colab.research.google.com/github/kimjiwoo2/Pill-agent/blob/develop/notebooks/%EC%A1%B0%EC%9C%A4%EC%88%98/pill_imprint_ocr_colab_0525.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pill Imprint OCR Colab Notebook

Pilliot 15k pill imprint OCR Colab notebook source.

흐름:
1. pilliot_15k_v1_final.zip 압축 해제
2. manifest 로드 및 OCR 후보 라벨 정리
3. Train/Inference skew를 막기 위한 공통 전처리와 학습 전용 증강 분리
4. Synthetic data 생성용 scaffold 제공
5. bbox 기준 알약 crop 및 OCR용 2차 전처리
6. EasyOCR / PaddleOCR baseline 실행
7. OCR + 속성 기반 DB 매핑 후보 점수화 scaffold
8. metrics와 실패 케이스 확인


## 0. Colab setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# PaddlePaddle을 반드시 먼저 설치해야 함.
# EasyOCR(PyTorch)보다 나중에 설치하면 NCCL 라이브러리 충돌 발생:
#   libtorch_cuda.so: undefined symbol: ncclCommShrink
!pip -q install paddlepaddle-gpu==3.1.0 -i \
    https://www.paddlepaddle.org.cn/packages/stable/cu118/
!pip -q install "paddleocr>=3.0.0"

# PaddlePaddle 설치 후 나머지 패키지 설치
!pip -q install easyocr opencv-python-headless pandas numpy matplotlib tqdm rapidfuzz pillow-heif
!pip -q install albumentations

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 GB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 51.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 699.9/699.9 MB 711.4 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 MB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from __future__ import annotations

import re
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import cv2
import numpy as np
import pandas as pd
from rapidfuzz.distance import Levenshtein
from tqdm.auto import tqdm


## 1. Paths


In [ ]:
# DATA_ROOT = Path("/content/pilliot_15k_v1_final")
# DATA_ROOT = Path("/content/drive/MyDrive/pillot/pilliot_15k_v1_final")

ZIP_PATH = "/content/drive/MyDrive/Pillot/dataset/pilliot_15k_v1_final.zip"
!unzip -q "{ZIP_PATH}" -d /content


In [ ]:
DATA_ROOT = Path("/content/pilliot_15k_v1_final")

MANIFEST_DIR = DATA_ROOT / "manifests"
IMAGE_ROOT = DATA_ROOT / "images"

WORK_DIR = Path("/content/pillot_ocr_work")
CROP_DIR = WORK_DIR / "crops"
RESULT_DIR = WORK_DIR / "results"

for directory in [WORK_DIR, CROP_DIR, RESULT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

## 2. Manifest loading


In [ ]:
def load_manifest(data_root: Path, manifest_name: str = "pilliot_15k_single_12k_manifest.csv") -> pd.DataFrame:
    """manifest CSV를 읽고 필수 컬럼이 있는지 확인합니다."""
    manifest_path = data_root / "manifests" / manifest_name
    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest not found: {manifest_path}")

    df = pd.read_csv(manifest_path)
    required = {"dataset_type", "split_type", "image_zip_name", "image_file"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in {manifest_path.name}: {sorted(missing)}")
    return df


# OCR 정답에서 비문자 라벨 제외
IGNORE_IMPRINT_TOKENS = {
    "",
    "NAN",
    "NONE",
    "NULL",
    "마크",
    "분할선",
    "없음",
    "무",
    "-",
}


def normalize_imprint(text: object, keep_separator: bool = False) -> str:
    """각인 라벨/예측값을 비교 가능한 형태로 정규화"""
    if pd.isna(text):
        return ""

    text = str(text).strip()
    if text.upper() in IGNORE_IMPRINT_TOKENS:
        return ""

    text = text.upper()
    text = re.sub(r"\s+", "", text)
    text = text.replace("분할선", "")


    allowed = r"[^0-9A-Z가-힣+\-/|]" if keep_separator else r"[^0-9A-Z가-힣+\-/]"
    text = re.sub(allowed, "", text)

    if text.upper() in IGNORE_IMPRINT_TOKENS:
        return ""
    return text


def build_target_candidates(row: pd.Series) -> list[str]:
    """
    이미지 한 장은 앞면 또는 뒷면 하나만 보임.
    따라서 print_front와 print_back 두 개 열을 후보 리스트로 관리
    """
    front = normalize_imprint(row.get("print_front", ""))
    back = normalize_imprint(row.get("print_back", ""))

    candidates = []
    for text in [front, back]:
        if text and text not in candidates:
            candidates.append(text)
    return candidates


def build_target_text(row: pd.Series) -> str:
    """화면 확인용 후보 문자열. 평가는 후보 중 하나와 맞는지로 처리"""
    return "/".join(build_target_candidates(row))


def normalize_prediction(text: object) -> str:
    return normalize_imprint(text, keep_separator=True)


def filter_ocr_candidates(df: pd.DataFrame, split: Optional[str] = None, limit: Optional[int] = None, seed: int = 42) -> pd.DataFrame:
    """OCR 평가 후보만 골라 target_candidates / target_text 컬럼을 추가"""
    out = df.copy()

    if "for_ocr_eval_candidate" in out.columns:
        out = out[out["for_ocr_eval_candidate"].astype(str).str.lower().eq("true")]
    if "has_print" in out.columns:
        out = out[out["has_print"].astype(str).str.lower().eq("true")]
    if split:
        out = out[out["split_type"].eq(split)]

    out["target_text_front"] = out["print_front"].map(normalize_imprint) if "print_front" in out.columns else ""
    out["target_text_back"] = out["print_back"].map(normalize_imprint) if "print_back" in out.columns else ""
    out["target_candidates"] = out.apply(build_target_candidates, axis=1)
    out["target_text"] = out["target_candidates"].map(lambda xs: "/".join(xs))
    out = out[out["target_text"].ne("")]

    if limit:
        import random
        random.seed(seed)
        groups = out.groupby("item_seq")
        all_kinds = list(groups.groups.keys())
        random.shuffle(all_kinds)

        selected, total = [], 0
        for kind in all_kinds:
            count = len(groups.get_group(kind))
            if total + count > limit:
                continue
            selected.append(kind)
            total += count
            if total >= limit * 0.9:
                break
        out = out[out["item_seq"].isin(selected)]

    return out.reset_index(drop=True)


df_all = load_manifest(DATA_ROOT)
df_eval = filter_ocr_candidates(df_all, split=None, limit=500)
print(df_all.shape, df_eval.shape)
display(df_eval.head(20))


(12000, 33) (477, 37)


,dataset_type,split_type,label_zip_name,json_file,image_file,dl_mapping_code,item_seq,dl_name,dl_company,drug_shape,...,has_print,sample_pack,sample_part,for_attribute,for_ocr_eval_candidate,for_detection,target_text_front,target_text_back,target_candidates,target_text
0,single,train,TL_76_단일.zip,K-035934_json/K-035934_0_2_0_0_75_060_200.json,K-035934_0_2_0_0_75_060_200.png,K-035934,201502216,더포르테씨정20mg/PTP,경동제약(주),육각형,...,True,zip12_max50_print_required,single_12k,True,True,True,CI,20,"[CI, 20]",CI/20
1,single,train,TL_76_단일.zip,K-035934_json/K-035934_0_2_0_0_75_220_200.json,K-035934_0_2_0_0_75_220_200.png,K-035934,201502216,더포르테씨정20mg/PTP,경동제약(주),육각형,...,True,zip12_max50_print_required,single_12k,True,True,True,CI,20,"[CI, 20]",CI/20
2,single,train,TL_76_단일.zip,K-035934_json/K-035934_0_2_0_0_75_260_200.json,K-035934_0_2_0_0_75_260_200.png,K-035934,201502216,더포르테씨정20mg/PTP,경동제약(주),육각형,...,True,zip12_max50_print_required,single_12k,True,True,True,CI,20,"[CI, 20]",CI/20
3,single,train,TL_76_단일.zip,K-035934_json/K-035934_0_2_0_0_75_300_200.json,K-035934_0_2_0_0_75_300_200.png,K-035934,201502216,더포르테씨정20mg/PTP,경동제약(주),육각형,...,True,zip12_max50_print_required,single_12k,True,True,True,CI,20,"[CI, 20]",CI/20
4,single,train,TL_76_단일.zip,K-035934_json/K-035934_0_2_0_0_75_320_200.json,K-035934_0_2_0_0_75_320_200.png,K-035934,201502216,더포르테씨정20mg/PTP,경동제약(주),육각형,...,True,zip12_max50_print_required,single_12k,True,True,True,CI,20,"[CI, 20]",CI/20
5,single,train,TL_76_단일.zip,K-035934_json/K-035934_0_2_0_0_90_080_200.json,K-035934_0_2_0_0_90_080_200.png,K-035934,201502216,더포르테씨정20mg/PTP,경동제약(주),육각형,...,True,zip12_max50_print_required,single_12k,True,True,True,CI,20,"[CI, 20]",CI/20
6,single,train,TL_76_단일.zip,K-035934_json/K-035934_0_2_0_0_90_260_200.json,K-035934_0_2_0_0_90_260_200.png,K-035934,201502216,더포르테씨정20mg/PTP,경동제약(주),육각형,...,True,zip12_max50_print_required,single_12k,True,True,True,CI,20,"[CI, 20]",CI/20
7,single,train,TL_76_단일.zip,K-035934_json/K-035934_0_2_0_0_90_280_200.json,K-035934_0_2_0_0_90_280_200.png,K-035934,201502216,더포르테씨정20mg/PTP,경동제약(주),육각형,...,True,zip12_max50_print_required,single_12k,True,True,True,CI,20,"[CI, 20]",CI/20
8,single,train,TL_76_단일.zip,K-035934_json/K-035934_0_2_0_0_90_320_200.json,K-035934_0_2_0_0_90_320_200.png,K-035934,201502216,더포르테씨정20mg/PTP,경동제약(주),육각형,...,True,zip12_max50_print_required,single_12k,True,True,True,CI,20,"[CI, 20]",CI/20
9,single,train,TL_76_단일.zip,K-035934_json/K-035934_0_2_0_1_75_060_200.json,K-035934_0_2_0_1_75_060_200.png,K-035934,201502216,더포르테씨정20mg/PTP,경동제약(주),육각형,...,True,zip12_max50_print_required,single_12k,True,True,True,CI,20,"[CI, 20]",CI/20


ex) 이미지에 CI or 20 둘 중 하나 인식하면 정답

## 3. Image path resolving


In [ ]:
def _zip_stem(name: str) -> str:
    return Path(str(name)).stem


def resolve_image_path(row: pd.Series, data_root: Path = DATA_ROOT) -> Path:
    """
    압축 해제 후 이미지 경로:
    pilliot_15k_v1_final/images/single/TS_43_단일/K-....png
    pilliot_15k_v1_final/images/combination/TS_8_조합/K-....png
    """
    dataset_type = str(row["dataset_type"])
    image_zip_stem = _zip_stem(row["image_zip_name"])
    image_file = str(row["image_file"])
    return data_root / "images" / dataset_type / image_zip_stem / image_file


def report_missing_images(df: pd.DataFrame, n: int = 20) -> pd.DataFrame:
    """manifest와 실제 이미지 파일 연결이 되는지 빠르게 확인"""
    paths = df.apply(resolve_image_path, axis=1)
    missing_mask = [not p.exists() for p in paths]
    missing = df.loc[missing_mask, ["dataset_type", "split_type", "image_zip_name", "image_file"]].copy()
    missing["expected_path"] = [str(p) for p, is_missing in zip(paths, missing_mask) if is_missing]
    print(f"missing images: {len(missing)} / {len(df)}")
    return missing.head(n)


display(report_missing_images(df_eval))


missing images: 0 / 477


,dataset_type,split_type,image_zip_name,image_file,expected_path


## 4. 공통 전처리


In [ ]:
# 설계 원칙:
# - 공통 전처리: 학습과 추론 모두 적용. 데이터 분포를 맞춰 Train-Inference Skew를 줄임
# - 학습 전용 증강: 공통 전처리 이후에만 적용.
# - 추론 전용 Reject: 사용자 입력 품질이 너무 낮을 때만 차단. training에서는 제외
# - CLAHE: 전체 이미지가 아니라, detection 후 crop된 알약에 대한 2차 OCR 전처리에서 수행

@dataclass
class CommonPreprocessConfig:
    """Detection 학습/추론에 공통으로 적용할 전처리 설정"""

    apply_awb: bool = True
    denoise_method: str = "bilateral"  # "none", "bilateral", "nlm"
    bilateral_d: int = 5
    bilateral_sigma_color: int = 35
    bilateral_sigma_space: int = 35
    nlm_h: int = 3
    target_size: int = 640
    pad_color: tuple[int, int, int] = (114, 114, 114)
    normalize_pixels: bool = False


@dataclass
class InferenceQualityConfig:
    """사용자 입력 이미지 reject 기준(학습 데이터에는 적용 X)"""

    min_laplacian_var: float = 35.0
    min_brightness: float = 35.0
    max_brightness: float = 230.0


def read_bgr(path: Path) -> np.ndarray:
    """
    이미지 포맷을 BGR 3채널 배열로 통일.
    PNG의 알파 채널은 제거되고, HEIC는 pillow-heif가 설치되어 있으면 읽을 수 있음.
    """
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix in {".heic", ".heif"}:
        try:
            import pillow_heif
            from PIL import Image

            pillow_heif.register_heif_opener()
            rgb = np.array(Image.open(path).convert("RGB"))
            return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        except ImportError as exc:
            raise ImportError("HEIC 이미지를 읽으려면 `pip install pillow-heif`가 필요합니다.") from exc

    image = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if image is None:
        raise FileNotFoundError(path)

    if image.ndim == 2:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    if image.shape[2] == 4:
        return cv2.cvtColor(image, cv2.COLOR_BGRA2BGR)
    return image


def gray_world_awb_bgr(image_bgr: np.ndarray) -> np.ndarray:
    """Gray World 방식의 간단한 오토 화이트 밸런스"""
    image = image_bgr.astype(np.float32)
    channel_means = image.reshape(-1, 3).mean(axis=0)
    gray_mean = channel_means.mean()
    scale = gray_mean / np.maximum(channel_means, 1e-6)
    balanced = image * scale
    return np.clip(balanced, 0, 255).astype(np.uint8)


def denoise_bgr(image_bgr: np.ndarray, cfg: CommonPreprocessConfig) -> np.ndarray:
    """
    OCR 각인을 지우지 않도록 약하게 노이즈 제거.
    강한 Gaussian Blur는 각인과 외곽선을 흐리게 만들 수 있어 사용하지 않음.
    """
    if cfg.denoise_method == "none":
        return image_bgr
    if cfg.denoise_method == "bilateral":
        return cv2.bilateralFilter(
            image_bgr,
            d=cfg.bilateral_d,
            sigmaColor=cfg.bilateral_sigma_color,
            sigmaSpace=cfg.bilateral_sigma_space,
        )
    if cfg.denoise_method == "nlm":
        return cv2.fastNlMeansDenoisingColored(image_bgr, None, cfg.nlm_h, cfg.nlm_h, 7, 21)
    raise ValueError(f"Unknown denoise_method: {cfg.denoise_method}")


def letterbox_bgr(
    image_bgr: np.ndarray,
    target_size: int = 640,
    pad_color: tuple[int, int, int] = (114, 114, 114),
) -> tuple[np.ndarray, float, tuple[int, int]]:
    """
    비율을 유지한 채 target_size 정사각형으로 맞춤.
    return: letterbox 이미지, resize 비율, 좌상단 padding(dx, dy)
    """
    h, w = image_bgr.shape[:2]
    scale = min(target_size / h, target_size / w)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))

    resized = cv2.resize(image_bgr, (new_w, new_h), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC)
    canvas = np.full((target_size, target_size, 3), pad_color, dtype=np.uint8)

    dx = (target_size - new_w) // 2
    dy = (target_size - new_h) // 2
    canvas[dy : dy + new_h, dx : dx + new_w] = resized
    return canvas, scale, (dx, dy)


def common_preprocess_bgr(
    image_bgr: np.ndarray,
    cfg: CommonPreprocessConfig = CommonPreprocessConfig(),
    do_letterbox: bool = False,
) -> np.ndarray | tuple[np.ndarray, float, tuple[int, int]]:
    """
    학습/추론 공통 전처리
    기본 반환은 BGR uint8이며, 모델 입력 직전에만 RGB/float 변환을 권장.
    """
    out = image_bgr
    if cfg.apply_awb:
        out = gray_world_awb_bgr(out)
    out = denoise_bgr(out, cfg)

    if do_letterbox:
        out, scale, pad = letterbox_bgr(out, cfg.target_size, cfg.pad_color)
        if cfg.normalize_pixels:
            out = out.astype(np.float32) / 255.0
        return out, scale, pad

    if cfg.normalize_pixels:
        out = out.astype(np.float32) / 255.0
    return out


def to_detection_input_rgb_float(image_bgr: np.ndarray) -> np.ndarray:
    """Detection 모델 입력 직전에만 RGB + 0~1 float로 변환"""
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    return rgb.astype(np.float32) / 255.0


def check_inference_quality(image_bgr: np.ndarray, cfg: InferenceQualityConfig = InferenceQualityConfig()) -> dict:
    """
    사용자 입력 이미지 품질 검증.
    학습에는 적용하지 않고, 실제 서비스 추론 전에만 reject 용도로 사용.
    """
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    blur_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())

    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    brightness = float(hsv[:, :, 2].mean())

    reasons = []
    if blur_score < cfg.min_laplacian_var:
        reasons.append("blur")
    if brightness < cfg.min_brightness:
        reasons.append("underexposed")
    if brightness > cfg.max_brightness:
        reasons.append("overexposed")

    return {
        "accept": len(reasons) == 0,
        "reasons": reasons,
        "blur_score": blur_score,
        "brightness": brightness,
    }


def build_training_augmentation():
    """
    학습 전용 증강(추론에는 절대 적용X)
    OCR 보존을 위해 Flip/Mirror 계열은 넣지 않는 것을 증강 원칙으로 함
    """
    import albumentations as A

    return A.Compose(
        [
            A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
            A.MotionBlur(blur_limit=3, p=0.15),
            A.GaussNoise(var_limit=(5.0, 25.0), p=0.25),
            A.CoarseDropout(
                max_holes=2,
                max_height=48,
                max_width=48,
                min_holes=1,
                fill_value=114,
                p=0.15,
            ),
            A.Rotate(limit=12, border_mode=cv2.BORDER_CONSTANT, value=(114, 114, 114), p=0.35),
        ],
        bbox_params=A.BboxParams(format="pascal_voc", label_fields=["class_labels"], min_visibility=0.2),
    )


def apply_training_augmentation(image_bgr: np.ndarray, bbox_xyxy: list[float]) -> tuple[np.ndarray, list[float]]:
    """
    학습 전용 증강 예시.
    입력 bbox는 [x1, y1, x2, y2] 형식이며, 추론 파이프라인에서는 호출하지 않음.
    """
    aug = build_training_augmentation()
    transformed = aug(
        image=image_bgr,
        bboxes=[bbox_xyxy],
        class_labels=["pill"],
    )
    new_bbox = list(transformed["bboxes"][0]) if transformed["bboxes"] else bbox_xyxy
    return transformed["image"], new_bbox


def preprocess_for_detection_train(
    image_bgr: np.ndarray,
    bbox_xyxy: Optional[list[float]] = None,
    cfg: CommonPreprocessConfig = CommonPreprocessConfig(),
    use_augmentation: bool = True,
) -> tuple[np.ndarray, Optional[list[float]]]:
    """
    Detection 학습 파이프라인
    공통 전처리 후 학습 전용 증강을 적용하고, 마지막에 모델 입력 크기로 letterbox 처리
    """
    image_bgr = common_preprocess_bgr(image_bgr, cfg, do_letterbox=False)

    if use_augmentation and bbox_xyxy is not None:
        image_bgr, bbox_xyxy = apply_training_augmentation(image_bgr, bbox_xyxy)

    image_bgr, scale, (dx, dy) = letterbox_bgr(image_bgr, cfg.target_size, cfg.pad_color)
    if bbox_xyxy is not None:
        x1, y1, x2, y2 = bbox_xyxy
        bbox_xyxy = [x1 * scale + dx, y1 * scale + dy, x2 * scale + dx, y2 * scale + dy]

    return image_bgr, bbox_xyxy


def preprocess_for_detection_inference(
    image_bgr: np.ndarray,
    cfg: CommonPreprocessConfig = CommonPreprocessConfig(),
    quality_cfg: InferenceQualityConfig = InferenceQualityConfig(),
) -> tuple[Optional[np.ndarray], dict]:
    """
    Detection 추론 파이프라인
    먼저 품질을 검사하고, 통과한 이미지만 공통 전처리 + letterbox를 적용
    """
    quality = check_inference_quality(image_bgr, quality_cfg)
    if not quality["accept"]:
        return None, quality

    image_bgr, scale, pad = common_preprocess_bgr(image_bgr, cfg, do_letterbox=True)
    quality["scale"] = scale
    quality["pad"] = pad
    return image_bgr, quality


## 5. bbox 기반 알약 crop 생성 및 2차 OCR 전처리


In [ ]:
@dataclass
class CropConfig:
    """OCR 입력 crop 생성 설정입니다."""

    margin_ratio: float = 0.16
    min_size: int = 96
    target_size: int = 384
    use_bbox: bool = True
    apply_common_preprocess: bool = True


def crop_with_bbox(image_bgr: np.ndarray, row: pd.Series, cfg: CropConfig) -> np.ndarray:
    """manifest bbox에 margin을 더해 알약 영역을 잘라냅니다."""
    h, w = image_bgr.shape[:2]
    if not cfg.use_bbox or any(pd.isna(row.get(k)) for k in ["bbox_x", "bbox_y", "bbox_w", "bbox_h"]):
        return image_bgr

    x, y, bw, bh = [float(row[k]) for k in ["bbox_x", "bbox_y", "bbox_w", "bbox_h"]]
    margin = cfg.margin_ratio * max(bw, bh)
    x1 = max(0, int(round(x - margin)))
    y1 = max(0, int(round(y - margin)))
    x2 = min(w, int(round(x + bw + margin)))
    y2 = min(h, int(round(y + bh + margin)))

    crop = image_bgr[y1:y2, x1:x2]
    if min(crop.shape[:2]) < cfg.min_size:
        return image_bgr
    return crop


def enhance_crop_for_ocr_bgr(image_bgr: np.ndarray, target_size: int = 384) -> np.ndarray:
    """
    OCR 전용 2차 전처리입니다.
    BGR crop을 grayscale 1채널로 바꾸고, CLAHE로 음각/양각 대비를 올립니다.
    EasyOCR/PaddleOCR 입력 호환을 위해 마지막에는 grayscale을 BGR 3채널로 복제합니다.
    """
    h, w = image_bgr.shape[:2]
    scale = target_size / max(h, w)
    if scale != 1:
        image_bgr = cv2.resize(image_bgr, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_CUBIC)

    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    return cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)


def make_crop(
    row: pd.Series,
    crop_cfg: CropConfig = CropConfig(),
    preprocess_cfg: CommonPreprocessConfig = CommonPreprocessConfig(),
) -> np.ndarray:
    """이미지 로드 → 공통 전처리 → bbox crop → OCR용 대비 보정 순서로 crop을 만듭니다."""
    path = resolve_image_path(row)
    image_bgr = read_bgr(path)

    if crop_cfg.apply_common_preprocess:
        image_bgr = common_preprocess_bgr(image_bgr, preprocess_cfg, do_letterbox=False)

    crop_bgr = crop_with_bbox(image_bgr, row, crop_cfg)
    return enhance_crop_for_ocr_bgr(crop_bgr, crop_cfg.target_size)


def save_eval_crops(df: pd.DataFrame, out_dir: Path = CROP_DIR, limit: Optional[int] = None) -> pd.DataFrame:
    """OCR 평가용 crop 이미지를 저장하고 crop_path를 manifest row에 붙입니다."""
    rows = df.head(limit).copy() if limit else df.copy()
    records = []

    for idx, row in tqdm(rows.iterrows(), total=len(rows), desc="cropping"):
        try:
            crop_bgr = make_crop(row)
            out_path = out_dir / f"{idx:06d}_{row['image_file']}"
            cv2.imwrite(str(out_path), crop_bgr)

            record = row.to_dict()
            record["crop_path"] = str(out_path)
            records.append(record)
        except Exception as exc:
            print(f"[skip] {row.get('image_file')}: {exc}")

    return pd.DataFrame(records)


df_crops = save_eval_crops(df_eval, limit=200)
display(df_crops[["image_file", "target_text_front", "target_text_back", "target_text", "crop_path"]].head())


cropping:   0%|          | 0/200 [00:00<?, ?it/s]

,image_file,target_text_front,target_text_back,target_text,crop_path
0,K-035934_0_2_0_0_75_060_200.png,CI,20,CI/20,/content/pillot_ocr_work/crops/000000_K-035934...
1,K-035934_0_2_0_0_75_220_200.png,CI,20,CI/20,/content/pillot_ocr_work/crops/000001_K-035934...
2,K-035934_0_2_0_0_75_260_200.png,CI,20,CI/20,/content/pillot_ocr_work/crops/000002_K-035934...
3,K-035934_0_2_0_0_75_300_200.png,CI,20,CI/20,/content/pillot_ocr_work/crops/000003_K-035934...
4,K-035934_0_2_0_0_75_320_200.png,CI,20,CI/20,/content/pillot_ocr_work/crops/000004_K-035934...


## 6. EasyOCR prototype


In [ ]:
def run_easyocr(df_crops: pd.DataFrame, gpu: bool = True) -> pd.DataFrame:
    """EasyOCR baseline입니다. crop_path 이미지를 읽어 OCR 결과를 저장"""
    import easyocr

    reader = easyocr.Reader(["en", "ko"], gpu=gpu)
    results = []

    for _, row in tqdm(df_crops.iterrows(), total=len(df_crops), desc="easyocr"):
        ocr = reader.readtext(
            row["crop_path"],
            detail=1,
            paragraph=False,
            batch_size=1,
            text_threshold=0.35,
            low_text=0.20,
            link_threshold=0.20,
            decoder="beamsearch",
        )

        pieces = []
        confs = []
        for _, text, conf in ocr:
            text = normalize_prediction(text)
            if text:
                pieces.append(text)
                confs.append(float(conf))

        record = row.to_dict()
        record["pred_text"] = "".join(pieces)
        record["ocr_conf"] = float(np.mean(confs)) if confs else 0.0
        record["raw_ocr"] = repr(ocr)
        results.append(record)

    out = pd.DataFrame(results)
    out.to_csv(RESULT_DIR / "easyocr_results.csv", index=False, encoding="utf-8-sig")
    return out


easyocr_results = run_easyocr(df_crops, gpu=True)
display(easyocr_results[["image_file", "target_text", "pred_text", "ocr_conf"]].head(20))


ImportError: /usr/local/lib/python3.12/dist-packages/torch/lib/libtorch_cuda.so: undefined symbol: ncclCommShrink

## 7. PaddleOCR stronger baseline


In [ ]:
# 재시작 후 실행 — 버전 확인
import paddle
print(paddle.__version__)        # 3.1.0 이어야 함
paddle.utils.run_check()         # OK 확인 후 진행

In [ ]:
# 모델 정의 및 추론
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from tqdm import tqdm

if "_PADDLE_OCR_MODEL" not in globals():
    _PADDLE_OCR_MODEL = None


def get_paddleocr_model():
    """
    PaddleOCR 3.x는 재초기화 시 리소스 충돌 가능성 있음.
    같은 런타임 내에서는 모델 객체를 한 번만 생성하고 재사용.
    """
    from paddleocr import PaddleOCR
    global _PADDLE_OCR_MODEL

    if _PADDLE_OCR_MODEL is not None:
        return _PADDLE_OCR_MODEL

    _PADDLE_OCR_MODEL = PaddleOCR(
        lang="korean",
        # 2.x의 use_angle_cls → 3.x에서는 use_textline_orientation으로 변경
        use_textline_orientation=True,
        # 문서 레벨 보정 기능 — 알약 단순 이미지에는 불필요하므로 비활성화
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        # det 파라미터는 3.x에서도 동일하게 지원
        det_db_box_thresh=0.25,
        det_db_unclip_ratio=1.8,
    )
    return _PADDLE_OCR_MODEL


def run_paddleocr(df_crops: pd.DataFrame) -> pd.DataFrame:
    """
    PaddleOCR 3.x 추론.
    - 2.x: ocr.ocr(path, cls=True)  → output[0] = [[[bbox], (text, conf)], ...]
    - 3.x: ocr.predict(path)         → output = [PredResult, ...]
           PredResult.rec_texts / PredResult.rec_scores 로 접근
    """
    ocr_model = get_paddleocr_model()

    results = []
    for _, row in tqdm(df_crops.iterrows(), total=len(df_crops), desc="paddleocr"):
        # 3.x API: predict() 사용, 반환값은 리스트[PredResult]
        output = ocr_model.predict(row["crop_path"])

        pieces = []
        confs = []

        for pred in (output or []):
            # 텍스트와 신뢰도를 rec_texts / rec_scores 속성으로 접근
            for text, conf in zip(pred.rec_texts, pred.rec_scores):
                text = normalize_prediction(text)
                if text:
                    pieces.append(text)
                    confs.append(float(conf))

        record = row.to_dict()
        record["pred_text"] = "".join(pieces)
        record["ocr_conf"] = float(np.mean(confs)) if confs else 0.0
        record["raw_ocr"] = repr(output)
        results.append(record)

    out = pd.DataFrame(results)
    out.to_csv(RESULT_DIR / "paddleocr_results.csv", index=False, encoding="utf-8-sig")
    return out


paddle_results = run_paddleocr(df_crops)
display(paddle_results[["image_file", "target_text", "pred_text", "ocr_conf"]].head(20))

## 9. Metrics and evaluation


In [ ]:
def char_accuracy(target: str, pred: str) -> float:
    """문자 단위 유사도입니다. 1.0이면 완전 일치입니다."""
    target = normalize_prediction(target)
    pred = normalize_prediction(pred)
    if not target:
        return float(pred == "")
    return max(0.0, 1.0 - Levenshtein.distance(target, pred) / len(target))


def parse_target_candidates(value: object) -> list[str]:
    """target_candidates 또는 target_text 문자열을 후보 리스트로 변환합니다."""
    if isinstance(value, list):
        return [normalize_prediction(v) for v in value if normalize_prediction(v)]
    if pd.isna(value):
        return []
    return [x for x in [normalize_prediction(part) for part in str(value).split("/")] if x]


def best_candidate_score(target_candidates: object, pred: str) -> tuple[str, bool, float]:
    """앞면/뒷면 후보 중 OCR 결과와 가장 가까운 후보를 찾습니다."""
    candidates = parse_target_candidates(target_candidates)
    pred = normalize_prediction(pred)
    if not candidates:
        return "", pred == "", float(pred == "")

    scored = [(target, target == pred, char_accuracy(target, pred)) for target in candidates]
    scored.sort(key=lambda x: (x[1], x[2]), reverse=True)
    return scored[0]


def evaluate_ocr_results(df: pd.DataFrame) -> dict:
    """OCR 결과를 후보 정답 기준으로 평가합니다."""
    exact = []
    chars = []

    for _, row in df.iterrows():
        pred = normalize_prediction(row["pred_text"])
        _, is_exact, score = best_candidate_score(row.get("target_candidates", row["target_text"]), pred)
        exact.append(is_exact)
        chars.append(score)

    return {
        "n": len(df),
        "exact_match": float(np.mean(exact)) if exact else 0.0,
        "char_accuracy": float(np.mean(chars)) if chars else 0.0,
    }


print("EasyOCR:", evaluate_ocr_results(easyocr_results))
# if "paddle_results" in globals():
#     print("PaddleOCR:", evaluate_ocr_results(paddle_results))


## 10. OCR 후처리 및 DB 매핑 scaffold


In [ ]:
OCR_CONFUSION_GROUPS = [
    ("0", "O"),
    ("1", "I", "L"),
    ("8", "B"),
]


def generate_ocr_text_variants(text: str, max_variants: int = 32) -> list[str]:
    """
    OCR에서 자주 헷갈리는 문자 후보를 생성합니다.
    실제 서비스에서는 무작정 치환하지 말고, DB 후보와 비교할 때만 보조 후보로 사용합니다.
    """
    base = normalize_prediction(text)
    variants = {base}

    for group in OCR_CONFUSION_GROUPS:
        next_variants = set(variants)
        for value in variants:
            for src in group:
                for dst in group:
                    if src != dst and src in value:
                        next_variants.add(value.replace(src, dst))
                        if len(next_variants) >= max_variants:
                            return sorted(next_variants)
        variants = next_variants

    return sorted(variants)


def text_similarity_with_variants(pred_text: str, target_text: str) -> float:
    """OCR 보정 후보 중 DB 각인과 가장 가까운 점수를 사용합니다."""
    target = normalize_prediction(target_text)
    if not target:
        return 0.0
    return max(char_accuracy(target, variant) for variant in generate_ocr_text_variants(pred_text))


def weighted_candidate_score(
    pred_text: str,
    candidate_row: pd.Series,
    pred_shape: Optional[str] = None,
    pred_color: Optional[str] = None,
    weights: dict[str, float] | None = None,
) -> float:
    """
    OCR + 속성 분류 결과를 합산해 DB 후보 점수를 계산하는 scaffold입니다.
    pred_shape/pred_color는 추후 속성 분류기 결과를 연결할 자리입니다.
    """
    weights = weights or {"ocr": 0.70, "shape": 0.15, "color": 0.15}

    front = normalize_imprint(candidate_row.get("print_front", ""))
    back = normalize_imprint(candidate_row.get("print_back", ""))
    imprint_score = max(
        [text_similarity_with_variants(pred_text, t) for t in [front, back] if t] or [0.0]
    )

    shape_score = 0.0
    if pred_shape and "drug_shape" in candidate_row:
        shape_score = float(str(candidate_row["drug_shape"]) == str(pred_shape))

    color_score = 0.0
    if pred_color and "color_class1" in candidate_row:
        color_candidates = {str(candidate_row.get("color_class1", "")), str(candidate_row.get("color_class2", ""))}
        color_score = float(str(pred_color) in color_candidates)

    return (
        weights["ocr"] * imprint_score
        + weights["shape"] * shape_score
        + weights["color"] * color_score
    )


def search_db_candidates(
    pred_text: str,
    db_df: pd.DataFrame,
    pred_shape: Optional[str] = None,
    pred_color: Optional[str] = None,
    top_k: int = 10,
) -> pd.DataFrame:
    """
    OCR 결과와 속성 분류 결과를 이용해 DB 후보 Top-K를 뽑습니다.
    현재는 manifest를 DB처럼 사용합니다. 추후 HIRA/의약품 DB로 교체하면 됩니다.
    """
    scored = db_df.copy()
    scored["candidate_score"] = scored.apply(
        lambda row: weighted_candidate_score(pred_text, row, pred_shape=pred_shape, pred_color=pred_color),
        axis=1,
    )
    cols = [
        "candidate_score",
        "dl_name",
        "item_seq",
        "print_front",
        "print_back",
        "drug_shape",
        "color_class1",
        "color_class2",
    ]
    cols = [c for c in cols if c in scored.columns]
    return scored.sort_values("candidate_score", ascending=False)[cols].head(top_k)


def build_structured_output(identified_rows: pd.DataFrame, image_quality: dict) -> dict:
    """RAG 에이전트에 넘길 수 있는 구조화 출력 예시입니다."""
    pills = []
    for _, row in identified_rows.iterrows():
        pills.append(
            {
                "pill_id": str(row.get("item_seq", "")),
                "name": row.get("dl_name", ""),
                "confidence": float(row.get("candidate_score", 0.0)),
            }
        )
    return {
        "identified_pills": pills,
        "metadata": {
            "image_quality": image_quality,
        },
    }


# 예시:
# search_db_candidates("TYLENOL", df_all, pred_shape="장방형", pred_color="하양", top_k=5)


## 11. CRAFT + PARSeq experiment skeleton


In [ ]:
"""
EasyOCR/PaddleOCR baseline이 부족하면 다음 단계로 넘어갑니다.

Detection:
  - CRAFT로 crop 내부 각인 영역을 찾습니다.
  - 또는 알약 bbox crop을 유지하고 각인 전용 detector를 따로 학습합니다.

Recognition:
  - PARSeq는 짧은 단어/토큰 인식에 강합니다.
  - pill imprint는 작은 문자, 음각/양각, 약한 대비가 많으므로 fine-tuning 후보입니다.
"""


def export_recognition_training_csv(df_crops: pd.DataFrame, out_path: Path = RESULT_DIR / "parseq_recognition_train.csv") -> Path:
    """
    PARSeq 학습용 CSV를 만듭니다.
    앞/뒤 후보가 모두 있는 이미지는 보이는 면을 확정할 수 없으므로 기본 제외합니다.
    """
    rec = df_crops[["crop_path", "target_text"]].copy()
    rec["target_candidates"] = df_crops["target_candidates"].map(parse_target_candidates)
    rec = rec[rec["target_candidates"].map(len).eq(1)].copy()
    rec["target_text"] = rec["target_candidates"].map(lambda xs: xs[0])
    rec["target_text"] = rec["target_text"].map(normalize_prediction)
    rec = rec[rec["target_text"].ne("")]
    rec[["crop_path", "target_text"]].to_csv(out_path, index=False, encoding="utf-8-sig")
    return out_path


parseq_csv = export_recognition_training_csv(df_crops)
print(parseq_csv)


## 12. Failure analysis helpers


In [ ]:
def add_error_columns(df: pd.DataFrame) -> pd.DataFrame:
    """OCR 결과표에 평가용 컬럼을 추가합니다."""
    out = df.copy()
    out["pred_norm"] = out["pred_text"].map(normalize_prediction)
    best = [best_candidate_score(t, p) for t, p in zip(out.get("target_candidates", out["target_text"]), out["pred_norm"])]
    out["matched_target"] = [x[0] for x in best]
    out["exact"] = [x[1] for x in best]
    out["char_acc"] = [x[2] for x in best]
    return out


def show_worst_cases(results: pd.DataFrame, n: int = 30) -> pd.DataFrame:
    """가장 안 맞은 케이스부터 확인합니다."""
    err = add_error_columns(results)
    cols = ["image_file", "target_text", "matched_target", "pred_text", "ocr_conf", "char_acc", "crop_path"]
    return err.sort_values(["exact", "char_acc", "ocr_conf"], ascending=[True, True, True])[cols].head(n)


display(show_worst_cases(easyocr_results, n=30))


## Appendix. Synthetic data scaffold


In [ ]:
def alpha_blend_rgba_on_bgr(
    background_bgr: np.ndarray,
    foreground_rgba: np.ndarray,
    x: int,
    y: int,
    shadow: bool = True,
) -> tuple[np.ndarray, list[int]]:
    """
    배경 위에 알약 RGBA 이미지를 합성하고 bbox를 자동 생성.
    합성 데이터는 shortcut learning을 줄이기 위해 알파 블렌딩과 약한 그림자를 함께 사용.
    """
    bg = background_bgr.copy()
    fg_bgr = cv2.cvtColor(foreground_rgba[:, :, :3], cv2.COLOR_RGB2BGR)
    alpha = foreground_rgba[:, :, 3].astype(np.float32) / 255.0

    h, w = alpha.shape
    bg_h, bg_w = bg.shape[:2]
    x1, y1 = max(0, x), max(0, y)
    x2, y2 = min(bg_w, x + w), min(bg_h, y + h)
    if x1 >= x2 or y1 >= y2:
        raise ValueError("foreground가 background 영역 밖에 있습니다.")

    fg_x1, fg_y1 = x1 - x, y1 - y
    fg_x2, fg_y2 = fg_x1 + (x2 - x1), fg_y1 + (y2 - y1)

    crop_alpha = alpha[fg_y1:fg_y2, fg_x1:fg_x2]
    crop_fg = fg_bgr[fg_y1:fg_y2, fg_x1:fg_x2]

    if shadow:
        shadow_mask = cv2.GaussianBlur(crop_alpha, (21, 21), 0)
        bg[y1:y2, x1:x2] = np.clip(bg[y1:y2, x1:x2] * (1.0 - 0.18 * shadow_mask[..., None]), 0, 255)

    bg_crop = bg[y1:y2, x1:x2].astype(np.float32)
    blended = crop_fg.astype(np.float32) * crop_alpha[..., None] + bg_crop * (1.0 - crop_alpha[..., None])
    bg[y1:y2, x1:x2] = np.clip(blended, 0, 255).astype(np.uint8)

    bbox_xyxy = [x1, y1, x2, y2]
    return bg, bbox_xyxy
